# SpineView — Cervical Landmark Detection Model

Train a lightweight keypoint detection model on the CSXA dataset for
accurate cervical lateral X-ray landmark detection.

**Output:** ONNX model (~15MB) that runs in-browser via ONNX Runtime Web.

## Setup
1. Download CSXA V3.0 from https://www.scidb.cn/en/detail?dataSetId=8e3b3d5e60a348ba961e19d48b881c90
2. Upload the zip to Google Drive
3. Run this notebook on Google Colab (GPU runtime)

In [ ]:
# ─── Install Dependencies ────────────────────────────────
!pip install -q torch torchvision timm albumentations opencv-python-headless onnx onnxruntime pillow tqdm

In [ ]:
# ─── Mount Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Set path to your uploaded CSXA dataset zip
DATASET_ZIP = '/content/drive/MyDrive/CSXA_V3.zip'  # <-- UPDATE THIS PATH
WORK_DIR = '/content/spine_training'
DATA_DIR = f'{WORK_DIR}/csxa'
OUTPUT_DIR = f'{WORK_DIR}/output'

In [ ]:
# ─── Extract Dataset ─────────────────────────────────────
import os, zipfile, shutil

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(DATA_DIR):
    print('Extracting CSXA dataset...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(WORK_DIR)
    # The zip may extract to a subfolder — find it
    extracted = [d for d in os.listdir(WORK_DIR) if os.path.isdir(f'{WORK_DIR}/{d}') and d != 'output']
    if extracted and not os.path.exists(DATA_DIR):
        os.rename(f'{WORK_DIR}/{extracted[0]}', DATA_DIR)
    print(f'Extracted to {DATA_DIR}')
else:
    print(f'Dataset already at {DATA_DIR}')

# Explore the structure
for root, dirs, files in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, '').count(os.sep)
    if depth < 2:
        indent = ' ' * 2 * depth
        print(f'{indent}{os.path.basename(root)}/')
        if depth == 0:
            for f in files[:10]:
                print(f'  {f}')
            if len(files) > 10:
                print(f'  ... and {len(files)-10} more files')

In [ ]:
# ─── Explore CSXA Annotation Format ──────────────────────
import json, glob

# Find JSON annotation files
json_files = glob.glob(f'{DATA_DIR}/**/*.json', recursive=True)
print(f'Found {len(json_files)} JSON files')

if json_files:
    # Look at the first annotation
    with open(json_files[0]) as f:
        sample = json.load(f)
    print('\nSample annotation structure:')
    print(json.dumps(sample, indent=2)[:3000])

In [ ]:
# ─── Data Loading & Mapping ──────────────────────────────
# This cell adapts to the CSXA annotation format.
# CSXA has 23 keypoints per image: 4 corners + centroid per
# vertebra (C2-C7) minus C1 which has fewer.
#
# We map CSXA keypoints → SpineView landmark IDs.

import json
import numpy as np
from pathlib import Path

# ── SpineView landmark order (must match posture-pro constants.ts) ──
SPINEVIEW_LANDMARKS = [
    'C1_post',       # C1 posterior tubercle
    'C2_sup_post',   # C2 superior-posterior
    'C2_inf_post',   # C2 inferior-posterior
    'C3_sup_post',   # C3 superior-posterior
    'C3_inf_post',   # C3 inferior-posterior
    'C4_sup_post',   # C4 superior-posterior
    'C4_inf_post',   # C4 inferior-posterior
    'C5_sup_post',   # C5 superior-posterior
    'C5_inf_post',   # C5 inferior-posterior
    'C6_sup_post',   # C6 superior-posterior
    'C6_inf_post',   # C6 inferior-posterior
    'C7_sup_post',   # C7 superior-posterior
    'C7_inf_post',   # C7 inferior-posterior
    'T1_sup_post',   # T1 superior-posterior
    'C2_sup_ant',    # C2 superior-anterior
    'C2_inf_ant',    # C2 inferior-anterior
    'C7_inf_ant',    # C7 inferior-anterior
]
NUM_KEYPOINTS = len(SPINEVIEW_LANDMARKS)
print(f'SpineView needs {NUM_KEYPOINTS} keypoints')

# ── Build the mapping from CSXA keypoint names → our indices ──
# This will be filled in after we inspect the CSXA format above.
# The CSXA paper says: 4 corner points + centroid per vertebra.
# Corner names are typically: sup_post, inf_post, sup_ant, inf_ant
#
# UPDATE THIS MAPPING based on the actual JSON keys you see above:

CSXA_TO_SPINEVIEW = {
    # Format: 'csxa_json_key': 'spineview_landmark_id'
    # Example (update after seeing real JSON keys):
    # 'C2_UL': 'C2_sup_post',
    # 'C2_LL': 'C2_inf_post',
    # 'C2_UR': 'C2_sup_ant',
    # 'C2_LR': 'C2_inf_ant',
}

print('\n⚠️  UPDATE CSXA_TO_SPINEVIEW mapping above after inspecting the JSON!')
print('Run the cell above first to see the annotation format.')

In [ ]:
# ─── Dataset Class ───────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

INPUT_SIZE = 384  # Model input resolution
HEATMAP_SIZE = 96  # Heatmap output resolution (INPUT_SIZE / 4)
SIGMA = 2.5       # Gaussian sigma for heatmap generation


def generate_heatmap(keypoints, heatmap_size, sigma):
    """Generate NUM_KEYPOINTS heatmaps from normalised keypoint coords."""
    heatmaps = np.zeros((NUM_KEYPOINTS, heatmap_size, heatmap_size), dtype=np.float32)
    for i, (x, y, vis) in enumerate(keypoints):
        if vis < 0.5:  # Not visible
            continue
        cx = int(x * heatmap_size)
        cy = int(y * heatmap_size)
        if cx < 0 or cx >= heatmap_size or cy < 0 or cy >= heatmap_size:
            continue
        # 2D Gaussian
        xx, yy = np.meshgrid(np.arange(heatmap_size), np.arange(heatmap_size))
        heatmaps[i] = np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
    return heatmaps


class CSXADataset(Dataset):
    def __init__(self, image_paths, annotations, transform=None):
        self.image_paths = image_paths
        self.annotations = annotations  # List of (N, 3) arrays: [x_norm, y_norm, visibility]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.image_paths[idx]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        kpts = self.annotations[idx].copy()  # (NUM_KEYPOINTS, 3)

        h_orig, w_orig = img.shape[:2]

        if self.transform:
            # Convert normalised coords to pixel coords for albumentations
            kpts_px = []
            for x, y, v in kpts:
                kpts_px.append((x * w_orig, y * h_orig, 0, 0))  # x, y, angle, scale

            transformed = self.transform(
                image=img,
                keypoints=kpts_px,
                keypoint_params=A.KeypointParams(format='xy', remove_invisible=False),
            )
            img = transformed['image']
            kpts_px_out = transformed['keypoints']

            # Convert back to normalised coords
            for i, (kx, ky, *_) in enumerate(kpts_px_out):
                kpts[i, 0] = kx / INPUT_SIZE
                kpts[i, 1] = ky / INPUT_SIZE
        else:
            img = cv2.resize(img, (INPUT_SIZE, INPUT_SIZE))
            img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        heatmaps = generate_heatmap(kpts, HEATMAP_SIZE, SIGMA)

        return img, torch.from_numpy(heatmaps), torch.from_numpy(kpts)


# ── Augmentation pipeline ──
train_transform = A.Compose(
    [
        A.Resize(INPUT_SIZE, INPUT_SIZE),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussNoise(var_limit=(5, 25), p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ],
    keypoint_params=A.KeypointParams(format='xy', remove_invisible=False),
)

val_transform = A.Compose(
    [
        A.Resize(INPUT_SIZE, INPUT_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ],
    keypoint_params=A.KeypointParams(format='xy', remove_invisible=False),
)

print(f'Dataset class ready. Input: {INPUT_SIZE}x{INPUT_SIZE}, Output heatmap: {HEATMAP_SIZE}x{HEATMAP_SIZE}')

In [ ]:
# ─── Load & Split Data ───────────────────────────────────
# This cell loads annotations, maps to SpineView landmarks,
# and creates train/val splits.

from sklearn.model_selection import train_test_split

def load_csxa_data(data_dir, mapping):
    """Load CSXA annotations and map to SpineView format."""
    image_paths = []
    annotations = []

    json_files = sorted(glob.glob(f'{data_dir}/**/*.json', recursive=True))
    print(f'Processing {len(json_files)} annotation files...')

    skipped = 0
    for jf in json_files:
        with open(jf) as f:
            ann = json.load(f)

        # Find corresponding image
        img_path = Path(jf).with_suffix('.png')
        if not img_path.exists():
            img_path = Path(jf).with_suffix('.jpg')
        if not img_path.exists():
            # Try looking in an images subfolder
            stem = Path(jf).stem
            candidates = list(Path(data_dir).rglob(f'{stem}.png')) + list(Path(data_dir).rglob(f'{stem}.jpg'))
            if candidates:
                img_path = candidates[0]
            else:
                skipped += 1
                continue

        # Read image dimensions
        img = Image.open(img_path)
        w, h = img.size

        # Map CSXA keypoints to SpineView landmarks
        kpts = np.zeros((NUM_KEYPOINTS, 3), dtype=np.float32)  # x_norm, y_norm, visibility
        mapped = 0
        for csxa_key, sv_id in mapping.items():
            sv_idx = SPINEVIEW_LANDMARKS.index(sv_id)
            if csxa_key in ann:
                pt = ann[csxa_key]
                if isinstance(pt, dict):
                    kpts[sv_idx] = [pt['x'] / w, pt['y'] / h, 1.0]
                elif isinstance(pt, (list, tuple)) and len(pt) >= 2:
                    kpts[sv_idx] = [pt[0] / w, pt[1] / h, 1.0]
                mapped += 1

        if mapped >= 8:  # Need at least 8 landmarks mapped
            image_paths.append(str(img_path))
            annotations.append(kpts)

    print(f'Loaded {len(image_paths)} samples (skipped {skipped} without images)')
    return image_paths, annotations


# Load data
assert len(CSXA_TO_SPINEVIEW) > 0, 'You need to fill in the CSXA_TO_SPINEVIEW mapping first!'

image_paths, annotations = load_csxa_data(DATA_DIR, CSXA_TO_SPINEVIEW)

# 85/15 train/val split
train_paths, val_paths, train_anns, val_anns = train_test_split(
    image_paths, annotations, test_size=0.15, random_state=42
)

train_ds = CSXADataset(train_paths, train_anns, transform=train_transform)
val_ds = CSXADataset(val_paths, val_anns, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# ─── Model: Lightweight HRNet-W18 ────────────────────────
import timm
import torch.nn as nn


class SpineLandmarkNet(nn.Module):
    """HRNet-W18 backbone with heatmap regression head."""

    def __init__(self, num_keypoints=NUM_KEYPOINTS, pretrained=True):
        super().__init__()
        # HRNet-W18 — lightweight, high-resolution representations
        self.backbone = timm.create_model(
            'hrnet_w18_small_v2',
            pretrained=pretrained,
            features_only=True,
        )

        # Get the output channels of the highest-resolution feature map
        with torch.no_grad():
            dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
            feats = self.backbone(dummy)
            feat_channels = feats[0].shape[1]
            feat_size = feats[0].shape[2]
            print(f'Backbone output: {feat_channels} channels, {feat_size}x{feat_size} spatial')

        # Heatmap head
        self.head = nn.Sequential(
            nn.Conv2d(feat_channels, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, num_keypoints, 1),
        )

        # Upsample to match target heatmap size if needed
        self.upsample = nn.Upsample(size=(HEATMAP_SIZE, HEATMAP_SIZE), mode='bilinear', align_corners=False)

    def forward(self, x):
        feats = self.backbone(x)
        heatmaps = self.head(feats[0])  # Use highest-resolution feature map
        heatmaps = self.upsample(heatmaps)
        return heatmaps


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = SpineLandmarkNet(NUM_KEYPOINTS).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params / 1e6:.1f}M')

In [ ]:
# ─── Training Loop ───────────────────────────────────────
from tqdm import tqdm

EPOCHS = 80
LR = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.MSELoss()

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_pck': []}


def pck_metric(pred_heatmaps, gt_keypoints, threshold=0.05):
    """Percentage of Correct Keypoints within threshold of normalised distance."""
    batch_size = pred_heatmaps.shape[0]
    correct = 0
    total = 0

    for b in range(batch_size):
        for k in range(NUM_KEYPOINTS):
            if gt_keypoints[b, k, 2] < 0.5:  # Not visible
                continue
            # Get predicted coordinate from heatmap argmax
            hm = pred_heatmaps[b, k]
            idx = hm.argmax()
            pred_y = (idx // HEATMAP_SIZE).float() / HEATMAP_SIZE
            pred_x = (idx % HEATMAP_SIZE).float() / HEATMAP_SIZE

            gt_x = gt_keypoints[b, k, 0]
            gt_y = gt_keypoints[b, k, 1]

            dist = ((pred_x - gt_x)**2 + (pred_y - gt_y)**2).sqrt()
            if dist < threshold:
                correct += 1
            total += 1

    return correct / max(total, 1)


for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    train_loss = 0
    for images, heatmaps, kpts in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        images = images.to(device)
        heatmaps = heatmaps.to(device)

        pred = model(images)
        loss = criterion(pred, heatmaps)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()

    # ── Validate ──
    model.eval()
    val_loss = 0
    val_pck = 0
    with torch.no_grad():
        for images, heatmaps, kpts in val_loader:
            images = images.to(device)
            heatmaps = heatmaps.to(device)

            pred = model(images)
            loss = criterion(pred, heatmaps)
            val_loss += loss.item()
            val_pck += pck_metric(pred.cpu(), kpts)

    val_loss /= len(val_loader)
    val_pck /= len(val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_pck'].append(val_pck)

    print(f'  Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | PCK@5%: {val_pck:.3f}')

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_model.pth')
        print(f'  ✓ Saved best model (val_loss={val_loss:.6f})')

print('\nTraining complete!')

In [ ]:
# ─── Training Curves ─────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.legend()
ax1.set_title('Loss')

ax2.plot(history['val_pck'], color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('PCK@5%')
ax2.set_title('Keypoint Accuracy')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
# ─── Export to ONNX ──────────────────────────────────────
import onnx

# Load best model
model.load_state_dict(torch.load(f'{OUTPUT_DIR}/best_model.pth'))
model.eval()
model.cpu()

# Export
dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
onnx_path = f'{OUTPUT_DIR}/spine_landmarks_cervical.onnx'

torch.onnx.export(
    model,
    dummy,
    onnx_path,
    opset_version=13,
    input_names=['image'],
    output_names=['heatmaps'],
    dynamic_axes={
        'image': {0: 'batch'},
        'heatmaps': {0: 'batch'},
    },
)

# Verify
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f'\n✓ ONNX model exported: {onnx_path}')
print(f'  Size: {size_mb:.1f} MB')
print(f'  Input: [1, 3, {INPUT_SIZE}, {INPUT_SIZE}]')
print(f'  Output: [1, {NUM_KEYPOINTS}, {HEATMAP_SIZE}, {HEATMAP_SIZE}]')

In [ ]:
# ─── Test ONNX Inference ─────────────────────────────────
import onnxruntime as ort

session = ort.InferenceSession(onnx_path)

# Run on a validation image
test_img, test_hm, test_kpts = val_ds[0]
pred = session.run(None, {'image': test_img.unsqueeze(0).numpy()})[0]

print(f'ONNX inference output shape: {pred.shape}')
print(f'Expected: (1, {NUM_KEYPOINTS}, {HEATMAP_SIZE}, {HEATMAP_SIZE})')

# Extract predicted coordinates
pred_coords = []
for k in range(NUM_KEYPOINTS):
    hm = pred[0, k]
    idx = hm.argmax()
    y = idx // HEATMAP_SIZE
    x = idx % HEATMAP_SIZE
    pred_coords.append((x / HEATMAP_SIZE, y / HEATMAP_SIZE))
    gt = test_kpts[k]
    if gt[2] > 0.5:
        dist = ((x/HEATMAP_SIZE - gt[0])**2 + (y/HEATMAP_SIZE - gt[1])**2)**0.5
        status = '✓' if dist < 0.05 else '✗'
        print(f'  {SPINEVIEW_LANDMARKS[k]:15s}: pred=({x/HEATMAP_SIZE:.3f}, {y/HEATMAP_SIZE:.3f}) gt=({gt[0]:.3f}, {gt[1]:.3f}) dist={dist:.4f} {status}')

In [ ]:
# ─── Visualise Predictions ───────────────────────────────
import matplotlib.pyplot as plt
import cv2

# Load original image
test_img_raw = cv2.imread(val_paths[0])
test_img_raw = cv2.cvtColor(test_img_raw, cv2.COLOR_BGR2RGB)
h, w = test_img_raw.shape[:2]

fig, ax = plt.subplots(1, 1, figsize=(8, 12))
ax.imshow(test_img_raw)

for k, (px, py) in enumerate(pred_coords):
    gt = test_kpts[k]
    if gt[2] > 0.5:
        # Ground truth (green)
        ax.plot(gt[0]*w, gt[1]*h, 'go', markersize=6, label='GT' if k == 0 else '')
        # Prediction (red)
        ax.plot(px*w, py*h, 'r+', markersize=8, label='Pred' if k == 0 else '')
        ax.annotate(SPINEVIEW_LANDMARKS[k].split('_')[0], (px*w, py*h),
                    fontsize=7, color='yellow', ha='left')

ax.legend()
ax.set_title('Landmark Predictions vs Ground Truth')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/prediction_sample.png', dpi=150)
plt.show()

In [ ]:
# ─── Save to Google Drive ────────────────────────────────
import shutil

drive_output = '/content/drive/MyDrive/SpineView_Model'
os.makedirs(drive_output, exist_ok=True)

shutil.copy(f'{OUTPUT_DIR}/spine_landmarks_cervical.onnx', drive_output)
shutil.copy(f'{OUTPUT_DIR}/best_model.pth', drive_output)
shutil.copy(f'{OUTPUT_DIR}/training_curves.png', drive_output)
shutil.copy(f'{OUTPUT_DIR}/prediction_sample.png', drive_output)

# Save landmark metadata for the browser runtime
metadata = {
    'model_name': 'spine_landmarks_cervical',
    'input_size': INPUT_SIZE,
    'heatmap_size': HEATMAP_SIZE,
    'num_keypoints': NUM_KEYPOINTS,
    'landmark_names': SPINEVIEW_LANDMARKS,
    'normalization': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225],
    },
}
with open(f'{drive_output}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\n✓ All files saved to Google Drive: {drive_output}')
print(f'\nFiles:')
for f in os.listdir(drive_output):
    size = os.path.getsize(f'{drive_output}/{f}') / (1024*1024)
    print(f'  {f} ({size:.1f} MB)')

print(f'\n🎉 Done! Download spine_landmarks_cervical.onnx and model_metadata.json')
print(f'   and place them in posture-pro/public/models/ for browser inference.')